# 4. 골든배치 1차 후보 강건성 검증

## 요약

기존 총수확량 기준 1차 후보 15개를 전략 내 표준화·부트스트랩·전략 층화 순열검정으로 다시 확인했다. DO 최솟값 상승과 pH 변동 감소 방향은 세 전략에서 일관됐지만, 8개 지표를 FDR 보정한 순열검정에서는 유의한 지표가 없었다. 확정 골든배치가 아니라 유망 후보로 유지한다. CSV는 저장하지 않는다.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


def find_project_root(start=Path.cwd()):
    for root in (start, *start.parents):
        if (root / 'data/interim/merged_data_ko.csv').exists():
            return root
    raise FileNotFoundError('merged_data_ko.csv를 찾을 수 없습니다.')


def fdr_bh(p_values):
    p_values = np.asarray(p_values, dtype=float)
    order = np.argsort(p_values)
    ranked = p_values[order]
    adjusted = ranked * len(ranked) / np.arange(1, len(ranked) + 1)
    adjusted = np.minimum.accumulate(adjusted[::-1])[::-1]
    result = np.empty_like(adjusted)
    result[order] = np.clip(adjusted, 0, 1)
    return result


candidate_ids = [8, 9, 12, 17, 26, 35, 52, 55, 57, 60, 63, 66, 79, 83, 85]
data = pd.read_csv(find_project_root() / 'data/interim/merged_data_ko.csv')
data = data.loc[data['배치번호'] <= 90].copy()
print('1차 후보:', candidate_ids)

### 판단

후보는 RC 5개, OC 5개, APC 5개다. 총수확량으로 선정된 후보이므로 총수확량을 다시 검정하지 않고 선정에 쓰지 않은 공정 안정성 지표만 평가한다.

In [ ]:
rows = []
for batch_number, batch in data.groupby('배치번호', sort=True):
    batch = batch.sort_values('발효시간(h)')
    time = batch['발효시간(h)'].to_numpy()
    late = time / time[-1] >= 0.8
    our = batch['산소소모율(g/min)'].to_numpy()
    rows.append({
        '배치번호': batch_number,
        '전략': 'RC' if batch_number <= 30 else ('OC' if batch_number <= 60 else 'APC'),
        '후보': batch_number in candidate_ids,
        'pH표준편차': batch['pH'].std(ddof=1),
        'DO최솟값': batch['용존산소(mg/L)'].min(),
        'OUR변동계수': our.std(ddof=1) / abs(our.mean()),
        'CO2최댓값': batch['배가스이산화탄소(%)'].max(),
        '기질후기기울기': np.polyfit(time[late], batch.loc[late, '기질농도(g/L)'], 1)[0],
        '온도표준편차': batch['발효온도(K)'].std(ddof=1),
        '산총투입량': np.trapezoid(batch['산투입유량(L/h)'], time),
        '염기총투입량': np.trapezoid(batch['염기투입유량(L/h)'], time),
    })
batch_metrics = pd.DataFrame(rows)
display(batch_metrics.groupby(['전략', '후보']).size().unstack())

### 판단

전략별 후보 5개와 비후보 25개 구성이 정확하다. 전략 자체의 값 차이가 후보 효과로 섞이지 않도록 모든 지표를 전략 안에서 표준화한다.

In [ ]:
metrics = ['pH표준편차', 'DO최솟값', 'OUR변동계수', 'CO2최댓값', '기질후기기울기', '온도표준편차', '산총투입량', '염기총투입량']
standardized = batch_metrics.copy()
for metric in metrics:
    standardized[metric] = standardized.groupby('전략')[metric].transform(
        lambda values: (values - values.mean()) / values.std(ddof=1)
    )

rng = np.random.default_rng(42)
permutation_count = 10000
permuted_labels = np.zeros((permutation_count, len(standardized)), dtype=bool)
for strategy in ['RC', 'OC', 'APC']:
    indices = np.flatnonzero(standardized['전략'].eq(strategy).to_numpy())
    random_scores = rng.random((permutation_count, len(indices)))
    selected_local = np.argpartition(random_scores, 5, axis=1)[:, :5]
    permuted_labels[np.arange(permutation_count)[:, None], indices[selected_local]] = True

result_rows = []
actual_labels = standardized['후보'].to_numpy()
for metric in metrics:
    values = standardized[metric].to_numpy()
    candidate_values = values[actual_labels]
    other_values = values[~actual_labels]
    observed_difference = candidate_values.mean() - other_values.mean()
    bootstrap_difference = (
        rng.choice(candidate_values, (5000, len(candidate_values)), replace=True).mean(axis=1)
        - rng.choice(other_values, (5000, len(other_values)), replace=True).mean(axis=1)
    )
    permuted_difference = (
        (permuted_labels @ values) / 15
        - ((~permuted_labels) @ values) / 75
    )
    permutation_p = (1 + np.sum(np.abs(permuted_difference) >= abs(observed_difference))) / (permutation_count + 1)
    result_rows.append({
        '지표': metric, '후보-비후보_z평균차': observed_difference,
        '부트스트랩95%CI_하한': np.quantile(bootstrap_difference, 0.025),
        '부트스트랩95%CI_상한': np.quantile(bootstrap_difference, 0.975),
        '층화순열_p': permutation_p,
    })
robust_results = pd.DataFrame(result_rows)
robust_results['순열_FDR'] = fdr_bh(robust_results['층화순열_p'])
display(robust_results.sort_values('순열_FDR').round(6))

direction_table = standardized.groupby(['전략', '후보'])[['DO최솟값', 'pH표준편차']].mean().unstack()
display(direction_table.round(6))

### 최종 판단

- 후보의 DO 최솟값은 비후보보다 전략 내 표준화 평균이 약 0.50 높고, pH 표준편차는 약 0.34 낮다. 두 방향은 RC·OC·APC 세 전략 모두에서 일관됐다.
- 단순 부트스트랩 신뢰구간은 DO 최솟값과 pH 변동에서 0을 제외하지만, 전략별로 후보 5개를 무작위 선정하는 층화 순열검정에 8개 FDR 보정을 적용하면 유의한 지표가 없다.
- 따라서 후보가 안정적인 방향을 보인다는 기술적 근거는 있으나, 우연한 후보 구성보다 확실히 좋다는 확증은 부족하다. 이전 Welch 결과보다 보수적인 결론이다.
- 이 목록은 총수확량으로 고른 1차 후보이며 확정 ID와 독립 검증 자료가 없다. 현재 상태의 적절한 판정은 ‘골든배치 확정’이 아니라 ‘후속 궤적 검토 대상 유지’다.